# Experiment: HIPS vs FTIR Intercept RH Recursive Diagnostics

This notebook targets the open issue from the meeting:

> Why does **HIPS vs FTIR EC** show a non-zero intercept when methods should ideally agree 1:1?

What this notebook does:

1. Recursively discovers local data files and rebuilds the ETAD merged dataset.
2. Quantifies baseline HIPS vs FTIR slope/intercept and residual structure.
3. Sweeps RH thresholds to test dry-vs-humid behavior changes.
4. Recursively tests other candidate drivers (temperature, wind, source terms, iron, etc.).
5. Reports what to check next, ranked by signal strength.



In [1]:
# Setup: imports, paths, recursive discovery

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / 'pyproject.toml').exists()), cwd)
notebook_dir = (repo_root / 'notebooks').resolve()
data_root = Path(
    os.environ.get('AETHMODULAR_DATA_ROOT', str(repo_root / 'research' / 'ftir_hips_chem'))
).expanduser().resolve()

scripts_path = (repo_root / 'research' / 'ftir_hips_chem' / 'scripts').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(scripts_path) not in sys.path:
    sys.path.insert(0, str(scripts_path))

from data_matching import (
    load_aethalometer_data,
    load_filter_data,
    add_base_filter_id,
    match_all_parameters,
    load_etad_factors_with_filter_ids,
)


def recursive_find(root: Path, pattern: str):
    return sorted(root.rglob(pattern))


def discover_data_files(repo_root: Path, data_root: Path):
    search_roots = [data_root, repo_root / 'research']
    found = {}

    for root in search_roots:
        if not root.exists():
            continue

        if 'filter_pkl' not in found:
            picks = [p for p in recursive_find(root, 'unified_filter_dataset.pkl') if 'metadata' not in p.name.lower()]
            if picks:
                found['filter_pkl'] = picks[0]

        if 'aeth_etad_pkl' not in found:
            candidates = recursive_find(root, 'df_Jacros_9am_resampled.pkl')
            if not candidates:
                candidates = [p for p in recursive_find(root, '*.pkl') if 'addis' in p.name.lower() or 'jacros' in p.name.lower()]
            if candidates:
                found['aeth_etad_pkl'] = candidates[0]

        if 'met_daily_csv' not in found:
            candidates = [
                p for p in recursive_find(root, '*.csv')
                if 'addis' in p.name.lower() and 'daily' in p.name.lower() and 'met' in p.name.lower()
            ]
            if candidates:
                found['met_daily_csv'] = candidates[0]

        if 'met_hourly_csv' not in found:
            candidates = [
                p for p in recursive_find(root, '*.csv')
                if 'meteostat' in str(p).lower() and 'master' in p.name.lower()
            ]
            if candidates:
                found['met_hourly_csv'] = candidates[0]

    return found


def get_season_3(month: int):
    if month in [10, 11, 12, 1, 2]:
        return 'Dry Season'
    if month in [3, 4, 5]:
        return 'Belg Rainy Season'
    if month in [6, 7, 8, 9]:
        return 'Kiremt Rainy Season'
    return None


FACTOR_TO_FRAC = {
    'GF3 (Charcoal)': 'charcoal_frac',
    'GF2 (Wood Burning)': 'wood_frac',
    'GF5 (Fossil Fuel Combustion)': 'fossil_fuel_frac',
    'GF4 (Polluted Marine)': 'polluted_marine_frac',
    'GF1 (Sea Salt Mixed)': 'sea_salt_frac',
}
FACTOR_TO_CONC = {
    'K_F3 Charcoal (ug/m3)': 'charcoal_conc',
    'K_F2 Wood Burning (ug/m3)': 'wood_conc',
    'K_F5 Fossil Fuel Combustion (ug/m3)': 'fossil_fuel_conc',
    'K_F4 Polluted Marine (ug/m3)': 'polluted_marine_conc',
    'K_F1 Sea Salt Mixed (ug/m3)': 'sea_salt_conc',
}


paths = discover_data_files(repo_root, data_root)
print(f'Repo root: {repo_root}')
print(f'Data root: {data_root}')
for key, path in paths.items():
    print(f'  {key}: {path}')

output_root = repo_root / 'artifacts' / 'notebook_outputs' / 'hips_ftir_intercept_rh_recursive'
plots_dir = output_root / 'plots'
tables_dir = output_root / 'tables'
plots_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
print(f'Output root: {output_root}')



Repo root: /Users/ahmadjalil/github/aethmodular
Data root: /Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem
  filter_pkl: /Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem/Filter Data/unified_filter_dataset.pkl
  aeth_etad_pkl: /Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem/df_Jacros_9am_resampled.pkl
  met_daily_csv: /Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem/Weather Data/Meteostat/Addis Ababa daily Average met Data.csv
  met_hourly_csv: /Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem/Weather Data/Meteostat/master_meteostat_AddisAbaba_63450_2022-12-01_2024-10-01.csv
Output root: /Users/ahmadjalil/github/aethmodular/artifacts/notebook_outputs/hips_ftir_intercept_rh_recursive


## Plan

- Build ETAD merged dataset (`ftir_ec`, `hips_fabs`, `ir_bcc`, meteorology, factors).
- Quantify baseline HIPS vs FTIR slope/intercept and residual structure.
- Run RH threshold sweep and identify best split(s).
- Recursively sweep additional variables to identify the next strongest confounder.
- Save figures/tables for follow-up checks.



In [ ]:
# Analysis: merge data, sweep RH, recursively rank next drivers

def resolve_met_path(*fallback_names):
    # Prefer explicitly discovered files, fallback to known relative locations.
    if 'met_daily_csv' in paths and any('daily' in x.lower() for x in fallback_names):
        return paths['met_daily_csv']
    if 'met_hourly_csv' in paths and any('master' in x.lower() for x in fallback_names):
        return paths['met_hourly_csv']

    for base in [data_root / 'Weather Data' / 'Meteostat', notebook_dir]:
        for name in fallback_names:
            candidate = base / name
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'Could not resolve any of: {fallback_names}')


def fit_line(df, x_col, y_col, min_n=5, force_zero=False):
    sub = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    n = len(sub)
    if n < min_n or sub[x_col].nunique() < 2:
        return {
            'n': n,
            'slope': np.nan,
            'intercept': np.nan,
            'r': np.nan,
            'r2': np.nan,
            'p': np.nan,
            'stderr': np.nan,
        }

    x = sub[x_col].to_numpy(dtype=float)
    y = sub[y_col].to_numpy(dtype=float)

    if force_zero:
        denom = np.dot(x, x)
        slope = np.dot(x, y) / denom if denom > 0 else np.nan
        intercept = 0.0
        y_hat = slope * x
        ss_res = np.sum((y - y_hat) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        # Through-origin fits use uncentered R² for a fair goodness-of-fit metric.
        ss_tot_unc = np.sum(y ** 2)
        r2 = 1 - ss_res / ss_tot_unc if ss_tot_unc > 0 else np.nan
        r = np.corrcoef(x, y)[0, 1] if np.std(x) > 0 and np.std(y) > 0 else np.nan
        p = np.nan
        stderr = np.nan
    else:
        slope, intercept, r, p, stderr = stats.linregress(x, y)
        r2 = r ** 2

    return {
        'n': n,
        'slope': float(slope),
        'intercept': float(intercept),
        'r': float(r),
        'r2': float(r2),
        'p': float(p) if pd.notna(p) else np.nan,
        'stderr': float(stderr) if pd.notna(stderr) else np.nan,
    }


def fit_ols(X: pd.DataFrame, y: pd.Series):
    XY = pd.concat([X, y.rename('y')], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    if XY.empty:
        return {'n': 0, 'coef': None, 'r2': np.nan, 'adj_r2': np.nan, 'rmse': np.nan}

    yv = XY['y'].to_numpy(dtype=float)
    Xv = XY.drop(columns='y').to_numpy(dtype=float)
    n, k = Xv.shape

    Xd = np.column_stack([np.ones(n), Xv])
    beta, *_ = np.linalg.lstsq(Xd, yv, rcond=None)
    y_hat = Xd @ beta

    ss_res = np.sum((yv - y_hat) ** 2)
    ss_tot = np.sum((yv - np.mean(yv)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k - 1) if n > k + 1 and pd.notna(r2) else np.nan
    rmse = float(np.sqrt(np.mean((yv - y_hat) ** 2)))

    return {
        'n': int(n),
        'coef': beta,
        'r2': float(r2),
        'adj_r2': float(adj_r2) if pd.notna(adj_r2) else np.nan,
        'rmse': rmse,
        'columns': ['intercept', *XY.drop(columns='y').columns.tolist()],
    }


# Load meteorology
met_daily = pd.read_csv(resolve_met_path(
    'Addis_Ababa_daily_Average_met_Data.csv',
    'Addis Ababa daily Average met Data.csv',
), encoding='utf-8-sig')
met_daily['date'] = pd.to_datetime(met_daily['date'])
met_daily['date_only'] = met_daily['date'].dt.normalize()

met_hourly = pd.read_csv(resolve_met_path(
    'master_meteostat_AddisAbaba_63450_2022-12-01_2024-10-01.csv',
))
met_hourly = met_hourly.replace('NA', np.nan)
met_hourly['time'] = pd.to_datetime(met_hourly['time'])
for col in ['temp', 'dwpt', 'rhum', 'prcp', 'wdir', 'wspd', 'pres']:
    if col in met_hourly.columns:
        met_hourly[col] = pd.to_numeric(met_hourly[col], errors='coerce')
met_hourly['date_only'] = met_hourly['time'].dt.normalize()

hourly_daily_agg = met_hourly.groupby('date_only').agg(
    rhum_mean=('rhum', 'mean'),
    temp_mean_hourly=('temp', 'mean'),
    dwpt_mean=('dwpt', 'mean'),
    wspd_mean=('wspd', 'mean'),
    wdir_mean=('wdir', 'mean'),
).reset_index()

# Load ETAD filter/aeth/factor merged data
factors_df = load_etad_factors_with_filter_ids().rename(columns={**FACTOR_TO_FRAC, **FACTOR_TO_CONC})
aethalometer_data = load_aethalometer_data()
filter_data = add_base_filter_id(load_filter_data())

df_aeth = aethalometer_data.get('Addis_Ababa')
bc_df = match_all_parameters('Addis_Ababa', 'ETAD', df_aeth, filter_data)

etad_filters = filter_data[filter_data['Site'] == 'ETAD'][['SampleDate', 'FilterId']].drop_duplicates()
etad_filters = etad_filters.rename(columns={'SampleDate': 'date', 'FilterId': 'base_filter_id'})
bc_df['date'] = pd.to_datetime(bc_df['date'])
etad_filters['date'] = pd.to_datetime(etad_filters['date'])

bc_with_id = pd.merge(bc_df, etad_filters, on='date', how='left')
merge_cols = ['base_filter_id', *FACTOR_TO_FRAC.values(), *FACTOR_TO_CONC.values()]
merge_cols = [c for c in merge_cols if c in factors_df.columns]

df = pd.merge(bc_with_id, factors_df[merge_cols], on='base_filter_id', how='left')
df['month'] = df['date'].dt.month
df['season'] = df['month'].apply(get_season_3)
df['date_only'] = df['date'].dt.normalize()

df = pd.merge(
    df,
    met_daily[['date_only', 'tavg', 'tmin', 'tmax', 'prcp']].copy(),
    on='date_only',
    how='left',
)
df = pd.merge(df, hourly_daily_agg, on='date_only', how='left')

df['abs_humidity_g_m3'] = (
    6.112 * np.exp((17.67 * df['tavg']) / (df['tavg'] + 243.5)) * df['rhum_mean'] * 2.1674 / (273.15 + df['tavg'])
)

if 'charcoal_conc' in df.columns and 'wood_conc' in df.columns:
    df['biomass_conc'] = df['charcoal_conc'].fillna(0) + df['wood_conc'].fillna(0)

# Primary dataset for intercept diagnostics
df_rh = df.dropna(subset=['ftir_ec', 'hips_fabs', 'rhum_mean']).copy()
print(f'Merged ETAD rows (ftir + hips + RH): {len(df_rh)}')
print(f'Date range: {df_rh["date"].min().date()} to {df_rh["date"].max().date()}')
print(f'RH range: {df_rh["rhum_mean"].min():.1f}% to {df_rh["rhum_mean"].max():.1f}%')

# Baseline fits
overall = fit_line(df_rh, 'ftir_ec', 'hips_fabs')
overall_zero = fit_line(df_rh, 'ftir_ec', 'hips_fabs', force_zero=True)

# Residual diagnostics
sub = df_rh[['ftir_ec', 'hips_fabs', 'rhum_mean']].dropna().copy()
sub['hips_pred'] = overall['slope'] * sub['ftir_ec'] + overall['intercept']
sub['hips_resid'] = sub['hips_fabs'] - sub['hips_pred']
resid_spearman = stats.spearmanr(sub['rhum_mean'], sub['hips_resid'], nan_policy='omit')

# RH threshold sweep
rh_thresholds = np.arange(30, 90, 5)
sweep_rows = []
for rh_t in rh_thresholds:
    lo = df_rh[df_rh['rhum_mean'] <= rh_t]
    hi = df_rh[df_rh['rhum_mean'] > rh_t]

    fit_lo = fit_line(lo, 'ftir_ec', 'hips_fabs')
    fit_hi = fit_line(hi, 'ftir_ec', 'hips_fabs')

    sweep_rows.append({
        'threshold': rh_t,
        'n_low': fit_lo['n'],
        'n_high': fit_hi['n'],
        'slope_low': fit_lo['slope'],
        'slope_high': fit_hi['slope'],
        'intercept_low': fit_lo['intercept'],
        'intercept_high': fit_hi['intercept'],
        'r2_low': fit_lo['r2'],
        'r2_high': fit_hi['r2'],
        'slope_gap': abs(fit_lo['slope'] - fit_hi['slope']) if pd.notna(fit_lo['slope']) and pd.notna(fit_hi['slope']) else np.nan,
        'distance_to_1_low': abs(fit_lo['slope'] - 1) if pd.notna(fit_lo['slope']) else np.nan,
        'distance_to_1_high': abs(fit_hi['slope'] - 1) if pd.notna(fit_hi['slope']) else np.nan,
    })

rh_sweep = pd.DataFrame(sweep_rows)

valid_low = rh_sweep[(rh_sweep['n_low'] >= 20) & rh_sweep['distance_to_1_low'].notna()]
best_low = valid_low.sort_values('distance_to_1_low').head(1)

valid_gap = rh_sweep[(rh_sweep['n_low'] >= 20) & (rh_sweep['n_high'] >= 20) & rh_sweep['slope_gap'].notna()]
best_gap = valid_gap.sort_values('slope_gap', ascending=False).head(1)

# Recursive scan for next strongest split variable
candidate_vars = [
    c for c in [
        'rhum_mean', 'abs_humidity_g_m3',
        'tavg', 'tmin', 'tmax', 'prcp', 'wspd_mean', 'wdir_mean',
        'iron', 'biomass_conc', 'fossil_fuel_conc',
        'charcoal_conc', 'wood_conc', 'polluted_marine_conc', 'sea_salt_conc',
    ] if c in df_rh.columns
]


def variable_split_scan(frame: pd.DataFrame, var: str, q_grid=np.arange(0.2, 0.81, 0.1), min_n=30):
    rows = []
    vals = frame[var].dropna()
    if vals.nunique() < 5:
        return pd.DataFrame(rows)

    for q in q_grid:
        thr = float(vals.quantile(q))
        lo = frame[frame[var] <= thr]
        hi = frame[frame[var] > thr]

        fit_lo = fit_line(lo, 'ftir_ec', 'hips_fabs', min_n=min_n)
        fit_hi = fit_line(hi, 'ftir_ec', 'hips_fabs', min_n=min_n)

        if fit_lo['n'] < min_n or fit_hi['n'] < min_n:
            continue

        rows.append({
            'variable': var,
            'quantile': q,
            'threshold': thr,
            'n_low': fit_lo['n'],
            'n_high': fit_hi['n'],
            'slope_low': fit_lo['slope'],
            'slope_high': fit_hi['slope'],
            'intercept_low': fit_lo['intercept'],
            'intercept_high': fit_hi['intercept'],
            'score_divergence': abs(fit_lo['slope'] - fit_hi['slope']) + 0.20 * abs(fit_lo['intercept'] - fit_hi['intercept']),
            'score_closest_to_1': min(abs(fit_lo['slope'] - 1), abs(fit_hi['slope'] - 1)),
        })

    return pd.DataFrame(rows)


scan_frames = [variable_split_scan(df_rh, v) for v in candidate_vars]
scan_frames = [x for x in scan_frames if not x.empty]
var_scan = pd.concat(scan_frames, ignore_index=True) if scan_frames else pd.DataFrame()

if not var_scan.empty:
    var_scan['coverage'] = (var_scan['n_low'] + var_scan['n_high']) / len(df_rh)
    var_scan['score_adjusted'] = var_scan['score_divergence'] * var_scan['coverage']

best_by_var = (
    var_scan.sort_values(['variable', 'score_adjusted'], ascending=[True, False])
    .groupby('variable', as_index=False)
    .first()
    .sort_values('score_adjusted', ascending=False)
) if not var_scan.empty else pd.DataFrame()

# Nested models: does adding RH help explain offset?
model_base = fit_ols(df_rh[['ftir_ec']], df_rh['hips_fabs'])
model_plus_rh = fit_ols(df_rh[['ftir_ec', 'rhum_mean']], df_rh['hips_fabs'])
model_plus_rh_inter = fit_ols(
    df_rh.assign(ftir_rh_inter=df_rh['ftir_ec'] * df_rh['rhum_mean'])[['ftir_ec', 'rhum_mean', 'ftir_rh_inter']],
    df_rh['hips_fabs'],
)

model_compare = pd.DataFrame([
    {
        'model': 'HIPS ~ FTIR',
        'n': model_base['n'],
        'r2': model_base['r2'],
        'adj_r2': model_base['adj_r2'],
        'rmse': model_base['rmse'],
    },
    {
        'model': 'HIPS ~ FTIR + RH',
        'n': model_plus_rh['n'],
        'r2': model_plus_rh['r2'],
        'adj_r2': model_plus_rh['adj_r2'],
        'rmse': model_plus_rh['rmse'],
    },
    {
        'model': 'HIPS ~ FTIR + RH + FTIR:RH',
        'n': model_plus_rh_inter['n'],
        'r2': model_plus_rh_inter['r2'],
        'adj_r2': model_plus_rh_inter['adj_r2'],
        'rmse': model_plus_rh_inter['rmse'],
    },
])

# Save tables
summary_baseline = pd.DataFrame([
    {
        'fit': 'OLS (free intercept)',
        **overall,
    },
    {
        'fit': 'Constrained through origin',
        **overall_zero,
    },
])

summary_baseline.to_csv(tables_dir / 'baseline_hips_vs_ftir.csv', index=False)
rh_sweep.to_csv(tables_dir / 'rh_threshold_sweep_hips_vs_ftir.csv', index=False)
model_compare.to_csv(tables_dir / 'nested_model_comparison.csv', index=False)
if not best_by_var.empty:
    best_by_var.to_csv(tables_dir / 'best_split_by_variable.csv', index=False)

# Diagnostic figure
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

# Panel A: scatter with RH color
ax = axes[0]
sc = ax.scatter(df_rh['ftir_ec'], df_rh['hips_fabs'], c=df_rh['rhum_mean'], cmap='viridis', s=50, alpha=0.8, edgecolors='black', linewidths=0.3)
lim = np.nanpercentile(np.r_[df_rh['ftir_ec'], df_rh['hips_fabs']], 99) * 1.1
x_line = np.linspace(0, lim, 200)
ax.plot(x_line, x_line, 'k--', alpha=0.6, label='1:1')
ax.plot(x_line, overall['slope'] * x_line + overall['intercept'], color='#D35400', lw=2.2,
        label=f"OLS: y={overall['slope']:.2f}x+{overall['intercept']:.2f}")
ax.plot(x_line, overall_zero['slope'] * x_line, color='#2C3E50', lw=1.8, linestyle=':',
        label=f"Zero-intercept: y={overall_zero['slope']:.2f}x")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel('FTIR EC (\u00b5g/m\u00b3)')
ax.set_ylabel('HIPS Fabs/MAC (\u00b5g/m\u00b3)')
ax.set_title('Overall HIPS vs FTIR')
ax.grid(alpha=0.3)
ax.legend(fontsize=9, loc='upper left')
plt.colorbar(sc, ax=ax, label='RH mean (%)')

# Panel B: RH threshold slope behavior
ax = axes[1]
ax.plot(rh_sweep['threshold'], rh_sweep['slope_low'], 'o-', color='#E67E22', label='Low RH (<= threshold)')
ax.plot(rh_sweep['threshold'], rh_sweep['slope_high'], 's-', color='#3498DB', label='High RH (> threshold)')
ax.axhline(1, color='green', linestyle='--', linewidth=1.5, alpha=0.6)
if not best_low.empty:
    b = best_low.iloc[0]
    ax.scatter([b['threshold']], [b['slope_low']], c='red', s=80, zorder=5)
    ax.annotate(f"best low: {int(b['threshold'])}%\n slope={b['slope_low']:.2f}",
                (b['threshold'], b['slope_low']), textcoords='offset points', xytext=(8, 8), fontsize=8)
ax.set_xlabel('RH threshold (%)')
ax.set_ylabel('Slope (HIPS vs FTIR)')
ax.set_title('RH Threshold Sweep')
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc='upper left')

# Panel C: residual vs RH
ax = axes[2]
ax.scatter(sub['rhum_mean'], sub['hips_resid'], c='#8E44AD', alpha=0.75, s=40, edgecolors='black', linewidths=0.2)
if sub['rhum_mean'].nunique() > 1:
    sl, ic, r_, p_, _ = stats.linregress(sub['rhum_mean'], sub['hips_resid'])
    x = np.linspace(sub['rhum_mean'].min(), sub['rhum_mean'].max(), 120)
    ax.plot(x, sl * x + ic, color='black', lw=2)
    ax.text(0.05, 0.95, f"resid~RH: slope={sl:.3f}, R\u00b2={r_**2:.2f}, p={p_:.3g}",
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.axhline(0, color='gray', linestyle=':', alpha=0.6)
ax.set_xlabel('RH mean (%)')
ax.set_ylabel('HIPS residual (obs - fitted)')
ax.set_title('Residual Pattern vs RH')
ax.grid(alpha=0.3)

plt.suptitle('HIPS Intercept Diagnostics: Baseline, RH Sweep, and Residual Structure', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
fig.savefig(plots_dir / 'hips_ftir_intercept_rh_diagnostics.png', dpi=220, bbox_inches='tight')
plt.show()

print('\nBaseline fit summary')
display(summary_baseline)

print('\nTop RH thresholds (closest low-RH slope to 1 and largest low/high gap)')
display(best_low)
display(best_gap)

print('\nNested model comparison')
display(model_compare)

if not best_by_var.empty:
    print('\nNext-variable ranking (best split per variable)')
    display(best_by_var.head(10))
else:
    print('\nNo variable splits passed minimum sample threshold.')


## Results

Run the analysis cell above first. Then use the findings block below for a concise interpretation and to decide the next test.



In [3]:
# Findings: concise interpretation + actionable next checks

# Pull key rows safely
best_low_row = best_low.iloc[0] if len(best_low) else None
best_gap_row = best_gap.iloc[0] if len(best_gap) else None

print('=' * 95)
print('INTERPRETATION SUMMARY (ETAD)')
print('=' * 95)

print(f"Overall OLS (HIPS vs FTIR): slope={overall['slope']:.3f}, intercept={overall['intercept']:.3f}, R²={overall['r2']:.3f}, n={overall['n']}")
print(f"Zero-intercept fit:          slope={overall_zero['slope']:.3f}, intercept=0.000, R²={overall_zero['r2']:.3f}, n={overall_zero['n']}")
print(f"Residual vs RH Spearman:     rho={resid_spearman.statistic:.3f}, p={resid_spearman.pvalue:.3g}")

if best_low_row is not None:
    print('-' * 95)
    print('Best low-RH threshold for approaching slope=1 (with n_low >= 20):')
    print(f"  RH <= {int(best_low_row['threshold'])}% -> slope={best_low_row['slope_low']:.3f}, intercept={best_low_row['intercept_low']:.3f}, n={int(best_low_row['n_low'])}")

if best_gap_row is not None:
    print('-' * 95)
    print('Largest low/high slope divergence (with n_low,n_high >= 20):')
    print(f"  threshold={int(best_gap_row['threshold'])}% | slope_low={best_gap_row['slope_low']:.3f} vs slope_high={best_gap_row['slope_high']:.3f}")
    print(f"  intercept_low={best_gap_row['intercept_low']:.3f} vs intercept_high={best_gap_row['intercept_high']:.3f}")
    print(f"  n_low={int(best_gap_row['n_low'])}, n_high={int(best_gap_row['n_high'])}")

print('-' * 95)
print('Model check (does adding RH help?):')
display(model_compare)

if not best_by_var.empty:
    print('-' * 95)
    print('Top candidate split variables after RH (best score per variable):')
    display(best_by_var.head(6))

    top_var = best_by_var.iloc[0]['variable']
    if top_var == 'rhum_mean':
        print('Recommended next thing: keep RH as primary confounder and test mechanism-specific corrections (drying/conditioning or RH-calibration).')
    else:
        print(f"Recommended next thing: run a focused mechanism check on '{top_var}' as secondary confounder after RH control.")

print('-' * 95)
print(f"Saved tables: {tables_dir}")
print(f"Saved plots:  {plots_dir}")



INTERPRETATION SUMMARY (ETAD)
Overall OLS (HIPS vs FTIR): slope=0.403, intercept=2.830, R²=0.785, n=378
Zero-intercept fit:          slope=0.865, intercept=0.000, R²=0.936, n=378
Residual vs RH Spearman:     rho=-0.210, p=3.73e-05
-----------------------------------------------------------------------------------------------
Best low-RH threshold for approaching slope=1 (with n_low >= 20):
  RH <= 45% -> slope=0.836, intercept=1.868, n=32
-----------------------------------------------------------------------------------------------
Largest low/high slope divergence (with n_low,n_high >= 20):
  threshold=45% | slope_low=0.836 vs slope_high=0.415
  intercept_low=1.868 vs intercept_high=2.734
  n_low=32, n_high=346
-----------------------------------------------------------------------------------------------
Model check (does adding RH help?):


,model,n,r2,adj_r2,rmse
0,HIPS ~ FTIR,378,0.785276,0.784705,0.483589
1,HIPS ~ FTIR + RH,378,0.800765,0.799703,0.465821
2,HIPS ~ FTIR + RH + FTIR:RH,378,0.800835,0.799238,0.465739


-----------------------------------------------------------------------------------------------
Top candidate split variables after RH (best score per variable):


,variable,quantile,threshold,n_low,n_high,slope_low,slope_high,intercept_low,intercept_high,score_divergence,score_closest_to_1,coverage,score_adjusted
4,iron,0.3,236.35200,113,264,0.458171,0.365926,2.393364,3.136032,0.240779,0.541829,0.997354,0.240142
12,wdir_mean,0.3,100.01250,114,264,0.520506,0.390250,2.418295,2.913017,0.229201,0.479494,1.000000,0.229201
7,rhum_mean,0.8,79.12500,303,75,0.405031,0.471764,2.845455,2.215770,0.192670,0.528236,1.000000,0.192670
0,abs_humidity_g_m3,0.8,11.38242,303,75,0.407450,0.468376,2.837778,2.245729,0.179335,0.531624,1.000000,0.179335
11,tmin,0.6,13.50000,232,146,0.377034,0.449737,2.985144,2.512959,0.167140,0.550263,1.000000,0.167140
14,wspd_mean,0.8,15.01500,302,76,0.394897,0.491124,2.873920,2.535656,0.163879,0.508876,1.000000,0.163879


Recommended next thing: run a focused mechanism check on 'iron' as secondary confounder after RH control.
-----------------------------------------------------------------------------------------------
Saved tables: /Users/ahmadjalil/github/aethmodular/artifacts/notebook_outputs/hips_ftir_intercept_rh_recursive/tables
Saved plots:  /Users/ahmadjalil/github/aethmodular/artifacts/notebook_outputs/hips_ftir_intercept_rh_recursive/plots


## Next Steps

1. Re-run with stricter RH bins (`<=40%`, `40-60%`, `>60%`) and compare intercept stability.
2. Test RH-corrected HIPS formulations in-place and re-check whether intercept collapses toward zero.
3. If RH is still dominant after correction, split by the top non-RH variable from `best_split_by_variable.csv`.

